In [1]:
import sys

print("Python version:")
print(sys.version)

import pandas as pd
import numpy as np
import sklearn

print("\nLibraries imported successfully!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

Python version:
3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]

Libraries imported successfully!
Pandas: 2.2.2
NumPy: 1.26.4
Scikit-learn: 1.5.1


In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../data/Blackout_PowerGraph_1000.csv")

# Display basic information
print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

Dataset shape: (1000, 225)

First 5 rows:


,bus_1_net_active_power,bus_2_net_active_power,bus_3_net_active_power,bus_4_net_active_power,bus_5_net_active_power,bus_6_net_active_power,bus_7_net_active_power,bus_8_net_active_power,bus_9_net_active_power,bus_10_net_active_power,...,line_30_rating,line_31_rating,line_32_rating,line_33_rating,line_34_rating,line_35_rating,line_36_rating,line_37_rating,line_38_rating,blackout
0,-0.033243,-0.019466,-0.228603,-0.095843,-0.092085,-0.173495,-0.033296,-0.217331,-0.222341,-0.247390,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0
1,-0.026204,-0.008625,-0.195121,-0.081933,-0.078730,-0.148137,-0.042461,-0.185511,-0.189782,-0.211138,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0
2,-0.034218,-0.020822,-0.222321,-0.093231,-0.089577,-0.168736,-0.034644,-0.211360,-0.216231,-0.240588,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0
3,-0.036552,-0.024058,-0.207475,-0.087072,-0.083664,-0.157496,-0.037863,-0.197252,-0.201796,-0.224513,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,1
4,0.029068,0.042205,-0.216479,-0.089889,-0.086306,-0.163932,0.040985,-0.205731,-0.210508,-0.234393,...,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0.356275,0



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Columns: 225 entries, bus_1_net_active_power to blackout
dtypes: float64(224), int64(1)
memory usage: 1.7 MB


In [3]:
# Count each class
class_counts = df["blackout"].value_counts().sort_index()

# Calculate percentages
class_percentages = df["blackout"].value_counts(normalize=True).sort_index() * 100

print("Class distribution:")
print(class_counts)

print("\nClass percentages:")
print(class_percentages.round(2))

print("\nMissing values:", df.isnull().sum().sum())

Class distribution:
blackout
0    798
1    202
Name: count, dtype: int64

Class percentages:
blackout
0    79.8
1    20.2
Name: proportion, dtype: float64

Missing values: 0


In [4]:
# Separate input features and target variable
X = df.drop(columns=["blackout"])
y = df["blackout"]

# Check duplicated input feature rows
duplicate_count = X.duplicated().sum()

print("Duplicated feature rows:", duplicate_count)

# Check whether duplicated features have consistent labels
duplicate_data = df[df.duplicated(
    subset=X.columns,
    keep=False
)]

if len(duplicate_data) > 0:
    label_counts = duplicate_data.groupby(
        list(X.columns)
    )["blackout"].nunique()

    print(
        "Duplicated feature groups with conflicting labels:",
        (label_counts > 1).sum()
    )
else:
    print("No duplicated feature rows found.")

Duplicated feature rows: 0
No duplicated feature rows found.


In [2]:
# Load the official group-aware split for the 1,000-record subset

import pandas as pd

DATA_PATH = "../data/Blackout_PowerGraph_1000.csv"
SPLIT_DIR = "../data/splits"

df = pd.read_csv(DATA_PATH)

# Reconstruct the record IDs used by the official audit
df.insert(
    0,
    "record_id",
    [f"small_{i:05d}" for i in range(len(df))]
)

# Load official manifests
train_manifest = pd.read_csv(f"{SPLIT_DIR}/train_manifest.csv")
validation_manifest = pd.read_csv(
    f"{SPLIT_DIR}/validation_manifest.csv"
)
test_manifest = pd.read_csv(f"{SPLIT_DIR}/test_manifest.csv")

# Select records belonging to the 1,000-record extract
train_ids = train_manifest.loc[
    train_manifest["source_extract"] == "1000_csv",
    "record_id"
].tolist()

validation_ids = validation_manifest.loc[
    validation_manifest["source_extract"] == "1000_csv",
    "record_id"
].tolist()

test_ids = test_manifest.loc[
    test_manifest["source_extract"] == "1000_csv",
    "record_id"
].tolist()

# Index the dataset by record ID
df_indexed = df.set_index("record_id")

feature_columns = [
    column for column in df.columns
    if column not in ["record_id", "blackout"]
]

# Construct the three datasets
X_train = df_indexed.loc[train_ids, feature_columns]
y_train = df_indexed.loc[train_ids, "blackout"]

X_val = df_indexed.loc[validation_ids, feature_columns]
y_val = df_indexed.loc[validation_ids, "blackout"]

X_test = df_indexed.loc[test_ids, feature_columns]
y_test = df_indexed.loc[test_ids, "blackout"]

# Display split sizes and class distributions
for name, labels in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")
    print("Records:", len(labels))
    print("Blackout cases:", int((labels == 1).sum()))
    print("Normal cases:", int((labels == 0).sum()))

print("\nFeature count:", X_train.shape[1])



Train
Records: 715
Blackout cases: 156
Normal cases: 559

Validation
Records: 161
Blackout cases: 28
Normal cases: 133

Test
Records: 124
Blackout cases: 18
Normal cases: 106

Feature count: 224


In [1]:
import os

print("Current directory:", os.getcwd())
print("\nData directory exists:", os.path.exists("../data"))
print("Splits directory exists:", os.path.exists("../data/splits"))
print("\nManifest files:")
if os.path.exists("../data/splits"):
    print(os.listdir("../data/splits"))

Current directory: /Users/caoyutong/Desktop/LAB6-GridSearch-and-Ensemble-Learning/notebooks

Data directory exists: True
Splits directory exists: True

Manifest files:
['test_manifest.csv', 'all_records_manifest.csv', 'train_manifest.csv', 'validation_manifest.csv']


In [6]:
# Check class distribution in each dataset
for name, labels in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name} set:")
    print(labels.value_counts().sort_index())
    print((labels.value_counts(normalize=True).sort_index() * 100).round(2))


Training set:
blackout
0    558
1    142
Name: count, dtype: int64
blackout
0    79.71
1    20.29
Name: proportion, dtype: float64

Validation set:
blackout
0    120
1     30
Name: count, dtype: int64
blackout
0    80.0
1    20.0
Name: proportion, dtype: float64

Test set:
blackout
0    120
1     30
Name: count, dtype: int64
blackout
0    80.0
1    20.0
Name: proportion, dtype: float64


In [3]:
from sklearn.tree import DecisionTreeClassifier

# Define the baseline model
baseline_model = DecisionTreeClassifier(random_state=42)

# Train using the official group-aware training set
baseline_model.fit(X_train, y_train)

print("Decision Tree baseline trained successfully.")
print("Training samples:", len(X_train))
print("Training features:", X_train.shape[1])
print("Tree depth:", baseline_model.get_depth())
print("Number of leaves:", baseline_model.get_n_leaves())


Decision Tree baseline trained successfully.
Training samples: 715
Training features: 224
Tree depth: 14
Number of leaves: 54


In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Predict on the validation set
y_val_pred = baseline_model.predict(X_val)
y_val_proba = baseline_model.predict_proba(X_val)[:, 1]

# Calculate validation metrics
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred, zero_division=0)
recall = recall_score(y_val, y_val_pred, zero_division=0)
f1 = f1_score(y_val, y_val_pred, zero_division=0)
roc_auc = roc_auc_score(y_val, y_val_proba)

print("========== VALIDATION RESULTS ==========")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["No Blackout", "Blackout"],
    zero_division=0
))


========== VALIDATION RESULTS ==========
Accuracy:  0.9255
Precision: 0.7353
Recall:    0.8929
F1-score:  0.8065
ROC-AUC:   0.9126

Confusion Matrix:
[[124   9]
 [  3  25]]

Classification Report:
              precision    recall  f1-score   support

 No Blackout       0.98      0.93      0.95       133
    Blackout       0.74      0.89      0.81        28

    accuracy                           0.93       161
   macro avg       0.86      0.91      0.88       161
weighted avg       0.93      0.93      0.93       161



In [5]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Evaluate the model on the training set
y_train_pred = baseline_model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(
    y_train, y_train_pred, zero_division=0
)
train_recall = recall_score(
    y_train, y_train_pred, zero_division=0
)
train_f1 = f1_score(
    y_train, y_train_pred, zero_division=0
)

print("========== TRAINING RESULTS ==========")
print(f"Accuracy:  {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall:    {train_recall:.4f}")
print(f"F1-score:  {train_f1:.4f}")

print("\n========== VALIDATION RESULTS ==========")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\nF1-score gap:")
print(f"{train_f1 - f1:.4f}")


========== TRAINING RESULTS ==========
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-score:  1.0000

========== VALIDATION RESULTS ==========
Accuracy:  0.9255
Precision: 0.7353
Recall:    0.8929
F1-score:  0.8065

F1-score gap:
0.1935


In [6]:

print("========== DECISION TREE COMPLEXITY ==========")

print("Tree depth:", baseline_model.get_depth())
print("Number of leaves:", baseline_model.get_n_leaves())
print("Number of nodes:", baseline_model.tree_.node_count)
print("Number of training samples:", len(X_train))
print("Number of input features:", X_train.shape[1])


========== DECISION TREE COMPLEXITY ==========
Tree depth: 14
Number of leaves: 54
Number of nodes: 107
Number of training samples: 715
Number of input features: 224


In [7]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Define controlled Decision Tree candidates
candidate_models = {
    "Original Baseline": baseline_model,
    "Controlled DT 1": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=5,
        random_state=42
    ),
    "Controlled DT 2": DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=10,
        random_state=42
    )
}

results = []

# Train and evaluate each model using training and validation data
for model_name, model in candidate_models.items():

    if model_name != "Original Baseline":
        model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)

    results.append({
        "Model": model_name,
        "Train F1": f1_score(y_train, y_train_pred, zero_division=0),
        "Validation Accuracy": accuracy_score(y_val, y_val_pred),
        "Validation Precision": precision_score(
            y_val, y_val_pred, zero_division=0
        ),
        "Validation Recall": recall_score(
            y_val, y_val_pred, zero_division=0
        ),
        "Validation F1": f1_score(
            y_val, y_val_pred, zero_division=0
        ),
        "Tree Depth": model.get_depth(),
        "Leaves": model.get_n_leaves()
    })

comparison_df = pd.DataFrame(results)

print("========== DECISION TREE COMPARISON ==========")
display(comparison_df.round(4))


========== DECISION TREE COMPARISON ==========


,Model,Train F1,Validation Accuracy,Validation Precision,Validation Recall,Validation F1,Tree Depth,Leaves
0,Original Baseline,1.0000,0.9255,0.7353,0.8929,0.8065,14,54
1,Controlled DT 1,0.8440,0.9068,0.7097,0.7857,0.7458,8,30
2,Controlled DT 2,0.8199,0.8571,0.5581,0.8571,0.6761,9,24


In [8]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Get validation probabilities from the original baseline
y_val_proba = baseline_model.predict_proba(X_val)[:, 1]

thresholds = [
    0.1, 0.2, 0.3, 0.4, 0.5,
    0.6, 0.7, 0.8, 0.9
]

threshold_results = []

for threshold in thresholds:

    y_val_threshold_pred = (
        y_val_proba >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_val_threshold_pred,
        labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, y_val_threshold_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, y_val_threshold_pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, y_val_threshold_pred, zero_division=0
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

threshold_df = pd.DataFrame(threshold_results)

print("========== VALIDATION THRESHOLD ANALYSIS ==========")
display(threshold_df.round(4))


========== VALIDATION THRESHOLD ANALYSIS ==========


,Threshold,Precision,Recall,F1-score,TN,FP,FN,TP
0,0.1,0.7353,0.8929,0.8065,124,9,3,25
1,0.2,0.7353,0.8929,0.8065,124,9,3,25
2,0.3,0.7353,0.8929,0.8065,124,9,3,25
3,0.4,0.7353,0.8929,0.8065,124,9,3,25
4,0.5,0.7353,0.8929,0.8065,124,9,3,25
5,0.6,0.7353,0.8929,0.8065,124,9,3,25
6,0.7,0.7353,0.8929,0.8065,124,9,3,25
7,0.8,0.7353,0.8929,0.8065,124,9,3,25
8,0.9,0.7353,0.8929,0.8065,124,9,3,25


In [9]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Final evaluation on the untouched test set
y_test_pred = baseline_model.predict(X_test)
y_test_proba = baseline_model.predict_proba(X_test)[:, 1]

print("========== FINAL TEST RESULTS ==========")

print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_test_proba):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["No Blackout", "Blackout"],
    zero_division=0
))


========== FINAL TEST RESULTS ==========
Accuracy:  0.8306
Precision: 0.4444
Recall:    0.6667
F1-score:  0.5333
ROC-AUC:   0.7626

Confusion Matrix:
[[91 15]
 [ 6 12]]

Classification Report:
              precision    recall  f1-score   support

 No Blackout       0.94      0.86      0.90       106
    Blackout       0.44      0.67      0.53        18

    accuracy                           0.83       124
   macro avg       0.69      0.76      0.71       124
weighted avg       0.87      0.83      0.84       124



In [10]:

error_analysis = pd.DataFrame({
    "record_id": X_test.index,
    "actual": y_test.values,
    "predicted": y_test_pred,
    "probability": y_test_proba
})

error_analysis["error_type"] = "Correct"
error_analysis.loc[
    (error_analysis["actual"] == 0) &
    (error_analysis["predicted"] == 1),
    "error_type"
] = "False Positive"

error_analysis.loc[
    (error_analysis["actual"] == 1) &
    (error_analysis["predicted"] == 0),
    "error_type"
] = "False Negative"

errors = error_analysis[
    error_analysis["error_type"] != "Correct"
].sort_values(
    by=["error_type", "probability"],
    ascending=[True, True]
)

print("========== TEST ERROR ANALYSIS ==========")
print(errors.to_string(index=False))

print("\nError counts:")
print(errors["error_type"].value_counts())


========== TEST ERROR ANALYSIS ==========
  record_id  actual  predicted  probability     error_type
small_00118       1          0          0.0 False Negative
small_00159       1          0          0.0 False Negative
small_00188       1          0          0.0 False Negative
small_00229       1          0          0.0 False Negative
small_00454       1          0          0.0 False Negative
small_00572       1          0          0.0 False Negative
small_00001       0          1          1.0 False Positive
small_00021       0          1          1.0 False Positive
small_00110       0          1          1.0 False Positive
small_00180       0          1          1.0 False Positive
small_00277       0          1          1.0 False Positive
small_00294       0          1          1.0 False Positive
small_00342       0          1          1.0 False Positive
small_00495       0          1          1.0 False Positive
small_00514       0          1          1.0 False Positive
small_00599   